<a href="https://colab.research.google.com/github/Andreas-Lukito/Stock_Sentiment_Analysis/blob/dev%2Fandreas/notebooks/01_text_cleaning_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# News Data Cleaning

## Import Libraries

In [13]:
!pip install python-dotenv cloudscraper newspaper3k tqdm contractions emoji lxml_html_clean

In [14]:
# Common Libraries
import numpy as np
import pandas as pd
import os
import sys

# Cleaner output
from tqdm import tqdm
from IPython.display import clear_output

# Google Colab Setup
from google.colab import drive
drive.mount('/content/drive')
project_path = "/content/drive/MyDrive/stock_news_sentiment_analysis"

## Add the path to text preprocessor
sys.path.append(os.path.abspath(os.path.join(project_path, "lib")))

# Text preprocessing
from preprocessor import clean_text
from scraper import extract_text_from_url, scrape_dataframe

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import the dataset

In [ ]:
news_data = pd.read_csv(os.path.join(project_path, "news_cache/catgorized_data/categorized_news_data2.csv"), sep=",")

##

In [ ]:
news_data.head()

## Imputing the `Text`

Since the previous method of fixing the contents of the article iesults in a more worse model, we will refetch the articles

In [ ]:
raw_rescraped_output_path = os.path.join(project_path, "news_cache/catgorized_data/categorized_news_data_rescraped_raw.csv")

news_data = scrape_dataframe(
    news_data,
    url_col="url",
    text_col="text",
    max_workers=5,
    batch_size=200,
    output_path=raw_rescraped_output_path,
)

## Text Cleaning

## Text Cleaning

In [ ]:

categorized_data_path = os.path.join(project_path,f"news_cache/catgorized_data/categorized_news_data_rescraped.csv")
categorized_data_path_folder = os.path.join(project_path,f"news_cache/catgorized_data/")
os.makedirs(categorized_data_path_folder, exist_ok=True)
overwrite_clean_data = True


In [ ]:
# tqdm for cleaner output
tqdm.pandas(desc="Cleaning the Text", unit="news")

# We will cache the data so that it will load faster
if os.path.exists(categorized_data_path) and not overwrite_clean_data:
    print("Loading cached dataset...")
    news_data = pd.read_csv(categorized_data_path)
    print("Cached dataset loaded")

elif os.path.exists(categorized_data_path) and overwrite_clean_data:
    print("Overwriting old data and caching new data...")
    # Clean the data
    news_data["clean_text"] = news_data["text"].progress_apply(
                                                        lambda x: clean_text(
                                                            text = x,
                                                            tokenize=False,
                                                            remove_stop_words= False, # Since we will be using a transformer model, we will not remove stop words since it can be useful for the model to understand the context of the sentence.
                                                            remove_emojis="keep"
                                                            )
                                                        )
    news_data.to_csv(categorized_data_path, index=False)
    print("Done Overwriting old data and caching new data...")

else:
    print("Creating and caching dataset...")
    # Clean the data
    news_data["clean_text"] = news_data["text"].progress_apply(
                                                        lambda x: clean_text(
                                                            text = x,
                                                            tokenize=False,
                                                            remove_stop_words= False,
                                                            remove_emojis="keep"
                                                            )
                                                        )
    news_data.to_csv(categorized_data_path, index=False)
    print("Finished Caching")